same as text file 

In [4]:
from pathlib import Path
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import cv2 as cv 
from sklearn.preprocessing import LabelEncoder 
from sklearn.model_selection import train_test_split

data_dir = Path("dataset/seg")
row = [] 
for class_d in sorted(data_dir.iterdir()) :
    if class_d.is_dir():
        for img_p in class_d.glob("*") :
            row.append({"path":str(img_p),"label":class_d.name})

df = pd.DataFrame(row) 

In [5]:
print ("num of img is ", len(df)) 
print (df["label"].unique()) 
print (df["label"].value_counts())

num of img is  14077
['buildings' 'forest' 'mountain' 'sea' 'street']
label
mountain     3037
street       2883
sea          2784
forest       2745
buildings    2628
Name: count, dtype: int64


In [6]:
df["label"].value_counts().plot(kind="pie" , autopct="%1.1f%%")
plt.show()

<Figure size 640x480 with 1 Axes>

In [7]:
sizes = []

for path in df["path"] : 
    img = cv.imread(path) 
    sizes.append(img.shape) 

df_sizes = pd.DataFrame(sizes , columns=["height", "width", "channel"]) 
print (df_sizes.value_counts().head(5))
print (df_sizes.describe())



height  width  channel
150     150    3          14043
113     150    3              7
131     150    3              2
144     150    3              2
141     150    3              2
Name: count, dtype: int64
             height    width  channel
count  14077.000000  14077.0  14077.0
mean     149.938765    150.0      3.0
std        1.520094      0.0      0.0
min       81.000000    150.0      3.0
25%      150.000000    150.0      3.0
50%      150.000000    150.0      3.0
75%      150.000000    150.0      3.0
max      150.000000    150.0      3.0


In [8]:
df_sample = (df.groupby("label")
             .sample(n=1000, random_state=42)
             .reset_index(drop=True)
             )

In [9]:
print(df_sample)
print (df_sample["label"].value_counts())

                                 path      label
0     dataset/seg/buildings/20805.jpg  buildings
1      dataset/seg/buildings/4233.jpg  buildings
2      dataset/seg/buildings/5791.jpg  buildings
3     dataset/seg/buildings/16750.jpg  buildings
4      dataset/seg/buildings/6235.jpg  buildings
...                               ...        ...
4995     dataset/seg/street/22035.jpg     street
4996     dataset/seg/street/11029.jpg     street
4997     dataset/seg/street/10448.jpg     street
4998     dataset/seg/street/22577.jpg     street
4999     dataset/seg/street/11470.jpg     street

[5000 rows x 2 columns]
label
buildings    1000
forest       1000
mountain     1000
sea          1000
street       1000
Name: count, dtype: int64


In [10]:

im_size= (64 , 64)

images= []
or_size = set() 

for pth in df_sample["path"]:
    img = cv.imread(pth)
    or_size.add(img.shape)

    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    img = cv.resize(img,im_size,interpolation=cv.INTER_AREA)
    img = img.astype(np.float32) / 255 

    images.append(img)

x = np.array(images)
le = LabelEncoder()
y = le.fit_transform(df_sample["label"])




In [11]:

x_train, x_temp, y_train, y_temp = train_test_split(x, y,test_size=.30 , random_state= 42 , stratify=y)

x_val, x_test , y_val, y_test = train_test_split(x_temp, y_temp,test_size=.50 , random_state= 42 , stratify=y_temp)

